# 02 — FinBERT Setup Test

**Purpose:** Verify the `finbert_env` environment is correctly installed and FinBERT scores text as expected.

**Run all four cells top to bottom before opening Notebook 03.**

---

## Cell 1 — Check library versions

In [1]:
import transformers
import torch
import pandas

print(f'transformers  : {transformers.__version__}   (expected 4.40.x)')
print(f'torch         : {torch.__version__}   (expected 2.2.x)')
print(f'pandas        : {pandas.__version__}   (expected 2.x)')
print(f'CUDA available: {torch.cuda.is_available()}  (False = CPU mode, this is fine)')
print()
print('✅  All libraries imported.')

transformers  : 4.40.0   (expected 4.40.x)
torch         : 2.2.2+cpu   (expected 2.2.x)
pandas        : 2.2.0   (expected 2.x)
CUDA available: False  (False = CPU mode, this is fine)

✅  All libraries imported.


C:\Users\Owner\AppData\Local\Temp\ipykernel_18992\4178286991.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas


## Cell 2 — Download and cache FinBERT

Downloads **ProsusAI/finbert** (~440 MB) from Hugging Face on first run.  
Subsequent runs load from local cache — **no internet needed after this.**

In [9]:
from transformers import BertTokenizer, BertForSequenceClassification

MODEL_NAME = 'ProsusAI/finbert'

print(f'Loading: {MODEL_NAME}')
print('First run downloads ~440 MB — may take a few minutes ...\n')

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model     = BertForSequenceClassification.from_pretrained(MODEL_NAME)

print('\n✅  Model loaded successfully.')
print(f'   Output labels : {model.config.id2label}')
# Expected: {0: 'positive', 1: 'negative', 2: 'neutral'}

Loading: ProsusAI/finbert
First run downloads ~440 MB — may take a few minutes ...


✅  Model loaded successfully.
   Output labels : {0: 'positive', 1: 'negative', 2: 'neutral'}


## Cell 3 — Score three test sentences

Each sentence should produce a clearly dominant class.  
All three must show ✅ before proceeding.

In [11]:
from transformers import pipeline

clf = pipeline(
    'text-classification',
    model='ProsusAI/finbert',
    return_all_scores=True
)

tests = [
    ('Central bank flags rising systemic risk in credit markets.',  'negative'),
    ('Economic growth remains robust and unemployment is falling.', 'positive'),
    ('The committee reviewed the quarterly inflation figures.',     'neutral'),
]

all_pass = True
for text, expected in tests:
    result = clf(text)[0]
    scores = {r['label']: round(r['score'], 4) for r in result}
    got    = max(scores, key=scores.get)
    ok     = '✅' if got == expected else '❌'
    if got != expected:
        all_pass = False
    print(f'{ok}  Expected: {expected:8s}  Got: {got:8s}  | {scores}')
    print(f'   Text: {text[:65]}')
    print()

print('=' * 70)
if all_pass:
    print('✅  ALL TESTS PASSED — proceed to Notebook 03.')
else:
    print('❌  A test failed. Re-run Cell 2 or check your transformers version.')

✅  Expected: negative  Got: negative  | {'positive': 0.0278, 'negative': 0.9239, 'neutral': 0.0482}
   Text: Central bank flags rising systemic risk in credit markets.

❌  Expected: positive  Got: negative  | {'positive': 0.0913, 'negative': 0.8895, 'neutral': 0.0192}
   Text: Economic growth remains robust and unemployment is falling.

✅  Expected: neutral   Got: neutral   | {'positive': 0.0331, 'negative': 0.1934, 'neutral': 0.7735}
   Text: The committee reviewed the quarterly inflation figures.

❌  A test failed. Re-run Cell 2 or check your transformers version.


## Cell 4 — Domain test with central bank language

Confirms FinBERT handles the hedged, indirect language typical of BIS speeches.

In [13]:
cb_tests = [
    'We remain vigilant to the build-up of vulnerabilities in the financial system.',
    'Credit conditions have tightened considerably and asset prices have declined sharply.',
    'The financial system remains well capitalised and resilient to adverse shocks.',
    'Uncertainty surrounding the global economic outlook has increased markedly.',
    'Monetary policy transmission is operating effectively through the banking channel.',
]

print('Central bank language domain test:')
print('-' * 75)
for s in cb_tests:
    result = clf(s)[0]
    scores = {r['label']: round(r['score'], 3) for r in result}
    top    = max(scores, key=scores.get)
    print(f'  [{top.upper():8s}]  pos={scores["positive"]:.3f}  '
          f'neg={scores["negative"]:.3f}  neu={scores["neutral"]:.3f}')
    print(f'             {s}')
    print()

Central bank language domain test:
---------------------------------------------------------------------------
  [NEUTRAL ]  pos=0.070  neg=0.226  neu=0.704
             We remain vigilant to the build-up of vulnerabilities in the financial system.

  [NEGATIVE]  pos=0.022  neg=0.961  neu=0.017
             Credit conditions have tightened considerably and asset prices have declined sharply.

  [POSITIVE]  pos=0.794  neg=0.013  neu=0.193
             The financial system remains well capitalised and resilient to adverse shocks.

  [POSITIVE]  pos=0.545  neg=0.414  neu=0.040
             Uncertainty surrounding the global economic outlook has increased markedly.

  [NEUTRAL ]  pos=0.056  neg=0.014  neu=0.929
             Monetary policy transmission is operating effectively through the banking channel.

